# TP 2 — Explorer et préparer un jeu de transactions
## Séance 2 : Données de transactions, exploration et préparation

| | |
|---|---|
| **École** | ESAIP |
| **Classe** | Aix IR 5 CP |
| **Cours** | Détection d’anomalies et fraude avec Python |
| **Enseignant** | Serigne Faye |

## Contexte

Votre banque vous confie un fichier de **10 040 transactions** par carte enregistrées sur trois mois (janvier à mars 2026). Une petite partie est frauduleuse. Avant toute modélisation, vous devez :

1. charger le fichier et le comprendre ;
2. repérer et corriger les problèmes de qualité ;
3. explorer le lien entre les variables et la fraude ;
4. préparer les données pour un futur modèle, **sans fuite de données**.

**Rappel des colonnes**

| Colonne | Contenu |
|---|---|
| `id_transaction`, `id_client` | identifiants |
| `date_heure` | date et heure de la transaction |
| `montant` | montant en euros |
| `categorie` | type de commerce |
| `canal` | carte présente, sans contact ou en ligne |
| `pays` | pays de la transaction |
| `fraude` | 1 = fraude, 0 = transaction normale (**cible**) |

## Étape 1 — Création du fichier

Exécutez la cellule suivante **sans la modifier**. Elle crée le fichier `transactions.csv` dans le dossier du notebook. Vous allez ensuite le charger comme s’il vous avait été fourni par la banque.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def generer_transactions(seed=7):
    rng = np.random.default_rng(seed)
    n = 10000
    n_clients = 500

    poids = np.array([1] * 6 + [4] * 16 + [2] * 2, dtype=float)
    poids /= poids.sum()

    categories = ['alimentation', 'restauration', 'transport', 'vetements',
                  'electronique', 'voyage', 'jeux_en_ligne']
    p_cat = [0.30, 0.22, 0.15, 0.12, 0.08, 0.06, 0.07]
    mu = {'alimentation': 3.3, 'restauration': 3.2, 'transport': 2.8, 'vetements': 4.0,
          'electronique': 5.0, 'voyage': 5.3, 'jeux_en_ligne': 3.0}

    categorie = rng.choice(categories, size=n, p=p_cat)
    montant = np.array([rng.lognormal(mu[c], 0.5) for c in categorie])
    heure = rng.choice(24, size=n, p=poids)
    canal = rng.choice(['carte_presente', 'sans_contact', 'en_ligne'], size=n, p=[0.45, 0.30, 0.25])
    canal[categorie == 'jeux_en_ligne'] = 'en_ligne'
    pays = rng.choice(['FR', 'ES', 'IT', 'DE', 'US', 'NG'], size=n, p=[0.90, 0.03, 0.03, 0.02, 0.015, 0.005])
    fraude = np.zeros(n, dtype=int)

    # 100 fraudes : la nuit, en ligne, à l'étranger, montants élevés
    idx = rng.choice(n, size=100, replace=False)
    fraude[idx] = 1
    heure[idx] = np.where(rng.random(100) < 0.8, rng.integers(0, 6, 100), rng.integers(6, 24, 100))
    categorie[idx] = rng.choice(['electronique', 'voyage', 'jeux_en_ligne'], size=100, p=[0.4, 0.3, 0.3])
    montant[idx] = rng.lognormal(5.2, 0.6, 100)
    canal[idx] = np.where(rng.random(100) < 0.9, 'en_ligne', 'carte_presente')
    pays[idx] = np.where(rng.random(100) < 0.6, rng.choice(['US', 'NG', 'ES', 'DE'], size=100), 'FR')

    jour = rng.integers(0, 90, n)
    minute = rng.integers(0, 60, n)
    date_heure = (pd.Timestamp('2026-01-01') + pd.to_timedelta(jour, unit='D')
                  + pd.to_timedelta(heure, unit='h') + pd.to_timedelta(minute, unit='m'))

    df = pd.DataFrame({
        'id_transaction': [f'T{i:05d}' for i in range(n)],
        'id_client': [f'C{c:03d}' for c in rng.integers(0, n_clients, n)],
        'date_heure': date_heure,
        'montant': montant.round(2),
        'categorie': categorie,
        'canal': canal,
        'pays': pays,
        'fraude': fraude,
    })

    # Achats exceptionnels légitimes (anomalies réelles, mais pas des fraudes)
    legit = rng.choice(df.index[df['fraude'] == 0], size=3, replace=False)
    df.loc[legit, 'montant'] = [3200.00, 4500.00, 5800.00]
    df.loc[legit, 'categorie'] = 'voyage'

    # Problèmes de qualité : erreurs de saisie, valeurs négatives
    non_fraude = df.index[df['fraude'] == 0].to_numpy()
    saisie = rng.choice(non_fraude, size=4, replace=False)
    df.loc[saisie, 'montant'] = 99999.99
    neg = rng.choice(np.setdiff1d(non_fraude, saisie), size=6, replace=False)
    df.loc[neg, 'montant'] = -df.loc[neg, 'montant']

    # Valeurs manquantes
    df.loc[rng.choice(n, size=100, replace=False), 'montant'] = np.nan
    df.loc[rng.choice(n, size=200, replace=False), 'canal'] = np.nan

    # Doublons exacts
    doublons = df.sample(40, random_state=1)
    df = pd.concat([df, doublons], ignore_index=True)
    df = df.sort_values('date_heure', kind='stable').reset_index(drop=True)
    return df

# Création du fichier transactions.csv dans le dossier du notebook
generer_transactions().to_csv('transactions.csv', index=False)
print('Fichier transactions.csv créé.')

## Étape 2 — Chargement et premier regard

Complétez les cellules (remplacez chaque `...`). Pensez à convertir la colonne `date_heure` en vraie date lors de la lecture.

In [ ]:
df = ...   # lire transactions.csv en convertissant la colonne date_heure en date
print(df.shape)

In [ ]:
...   # afficher les 5 premières lignes

In [ ]:
...   # afficher les types des colonnes et le nombre de valeurs non nulles

In [ ]:
...   # statistiques descriptives

## Étape 3 — Contrôle de la qualité des données

Cherchez les trois types de problèmes vus en cours : valeurs manquantes, doublons, valeurs aberrantes.

In [ ]:
# 1. Valeurs manquantes : nombre et pourcentage par colonne
manquantes = ...
print(manquantes)
print((manquantes / len(df) * 100).round(2))

# 2. Doublons exacts
nb_doublons = ...
print('Doublons :', nb_doublons)

# 3. Montants négatifs ou nuls
nb_negatifs = ...
print('Montants <= 0 :', nb_negatifs)

# 4. Les 10 plus grands montants (colonnes montant, categorie, fraude)
...

## Étape 4 — Nettoyage

Règles à appliquer, dans cet ordre :

1. supprimer les doublons exacts ;
2. considérer comme **erreurs** les montants négatifs ou nuls et les montants supérieurs ou égaux à 99 999 € (code de saisie) : les remplacer par une valeur manquante ;
3. remplacer les montants manquants par la **médiane de la catégorie** ;
4. remplacer les canaux manquants par `'inconnu'`.

Les achats de plusieurs milliers d’euros qui restent plausibles ne sont **pas** des erreurs : on les garde.

In [ ]:
# 1. Supprimer les doublons
df = ...

# 2. Montants négatifs ou nuls, ou >= 99999 : remplacés par une valeur manquante
masque_erreur = ...
df.loc[masque_erreur, 'montant'] = np.nan

# 3. Montants manquants : médiane de la catégorie
#    (indice : df.groupby('categorie')['montant'].transform('median'))
df['montant'] = ...

# 4. Canal manquant : 'inconnu'
df['canal'] = ...

# Vérification
print(df.shape)
print('Valeurs manquantes restantes :', df.isna().sum().sum())

## Étape 5 — Exploration : quelles variables sont liées à la fraude ?

In [ ]:
# Taux de fraude global
taux_global = ...
print(f'Taux de fraude global : {taux_global:.2%}')

# Moyenne et médiane du montant
print('Montant moyen :', ...)
print('Montant médian :', ...)

# Variables dérivées
df['heure'] = ...                       # heure extraite de date_heure
df['nuit'] = ...                        # 1 si heure < 6, sinon 0
df['etranger'] = ...                    # 1 si pays différent de 'FR', sinon 0

**Taux de fraude par catégorie**

In [ ]:
# Taux de fraude (moyenne de la colonne fraude) par groupe, du plus élevé au plus faible
...   # par categorie

**Taux de fraude par canal**

In [ ]:
...   # par canal

**Taux de fraude par pays**

In [ ]:
...   # par pays

**Taux de fraude selon la nuit (heure < 6)**

In [ ]:
...   # par nuit (0 / 1)

**Taux de fraude selon le pays (étranger ou non)**

In [ ]:
...   # par etranger (0 / 1)

## Étape 6 — Visualisation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))

# Histogramme du montant en échelle linéaire
...

# Histogramme du montant en échelle logarithmique
# (indice : bins = np.logspace(np.log10(min), np.log10(max), 50) et ax[1].set_xscale('log'))
...

ax[0].set_title('Échelle linéaire')
ax[1].set_title('Échelle logarithmique')
plt.show()

In [ ]:
# Diagramme en barres du taux de fraude (en %) selon l'heure
taux_heure = ...
plt.figure(figsize=(10, 3.2))
...
plt.xlabel('Heure')
plt.ylabel('Taux de fraude (%)')
plt.xticks(range(24))
plt.show()

In [ ]:
# Boîtes à moustaches du montant selon la classe (fraude 0 / 1), échelle logarithmique
plt.figure(figsize=(5, 4))
...
plt.yscale('log')
plt.ylabel('Montant (€)')
plt.show()

## Étape 7 — Préparation des variables sans fuite de données

On découpe **dans le temps** : entraînement = transactions avant le 2 mars 2026, test = à partir du 2 mars 2026. Toutes les transformations apprises (moyenne, écart-type) sont calculées sur l’entraînement uniquement.

In [ ]:
# 1. Découpage temporel
train = ...   # transactions avant le 2026-03-02
test = ...    # transactions à partir du 2026-03-02
print('Entraînement :', len(train), 'lignes,', train['fraude'].sum(), 'fraudes')
print('Test         :', len(test), 'lignes,', test['fraude'].sum(), 'fraudes')

In [ ]:
# 2. Standardisation du montant : paramètres calculés sur l'ENTRAÎNEMENT uniquement
mu = ...
sigma = ...
train['montant_std'] = ...
test['montant_std'] = ...    # mêmes mu et sigma !
print(f'mu = {mu:.2f}   sigma = {sigma:.2f}')
print('Entraînement : moyenne =', round(train['montant_std'].mean(), 3), ' écart-type =', round(train['montant_std'].std(), 3))
print('Test         : moyenne =', round(test['montant_std'].mean(), 3), ' écart-type =', round(test['montant_std'].std(), 3))

In [ ]:
# 3. Encodage one-hot de categorie et canal (pd.get_dummies, dtype=int)
colonnes = ['montant_std', 'heure', 'nuit', 'etranger', 'categorie', 'canal']
X_train = ...
X_test = ...

# Le test doit avoir exactement les mêmes colonnes que l'entraînement
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train['fraude']
y_test = test['fraude']
print(X_train.shape, X_test.shape)
X_train.head()

## Questions

Répondez dans les cellules de texte situées sous chaque question.

**Question 1.** Combien de lignes et de colonnes contient le fichier ? Classez chaque colonne : identifiant, temporelle, numérique, catégorielle ou cible.

*Votre réponse :*

**Question 2.** Quels problèmes de qualité avez-vous trouvés ? Donnez le nombre pour chacun (valeurs manquantes, doublons, montants négatifs, montants extrêmes).

*Votre réponse :*

**Question 3.** Parmi les montants très élevés, lesquels avez-vous traités comme des erreurs et lesquels avez-vous gardés ? Justifiez. Pourquoi imputer par la médiane de la *catégorie* plutôt que par la médiane de tous les montants ?

*Votre réponse :*

**Question 4.** Que révèlent la moyenne et la médiane du montant ? Pourquoi utiliser l’échelle logarithmique ? Quel est le taux de fraude global et que cela implique-t-il pour l’évaluation d’un modèle ?

*Votre réponse :*

**Question 5.** Quelles variables semblent les plus liées à la fraude ? Appuyez-vous sur les taux calculés. Cela prouve-t-il que la variable *cause* la fraude ?

*Votre réponse :*

**Question 6.** Pourquoi calcule-t-on la moyenne et l’écart-type sur l’entraînement uniquement ? Pourquoi la moyenne et l’écart-type standardisés du test ne valent-ils pas exactement 0 et 1 ? Pourquoi découper dans le temps plutôt qu’au hasard ?

*Votre réponse :*

## Pour aller plus loin (facultatif)

1. Remplacez la standardisation manuelle par `StandardScaler` de scikit-learn (`fit` sur l’entraînement, `transform` sur l’entraînement et le test). Retrouvez-vous les mêmes valeurs ?
2. Appliquez `np.log1p` au montant avant la standardisation. Comment change l’histogramme ?
3. Calculez, pour chaque client, son nombre de transactions et son montant moyen. Les clients ayant subi une fraude se distinguent-ils des autres ?